# Практична робота №5 — Лінійне програмування: симплекс-метод, двоїстість

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твої коефіцієнти прибутку, матрицю витрат ресурсів і запаси автоматично.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import linprog


In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"


## 1. Побудова моделі

З таблиці варіантів дістаємо коефіцієнти прибутку `c`, матрицю витрат ресурсів `A` (рядки — сировина, машинний час, робочий час) та запаси `b`. Задача: Z = c1*x1 + c2*x2 -> max за Ax <= b, x >= 0.

In [ ]:
c_raw, A_raw, b_raw = VARIANTS[VARIANT]
c = np.array(c_raw, dtype=float)
A = np.array(A_raw, dtype=float)
b = np.array(b_raw, dtype=float)
n, m = 2, 3
RESOURCE_NAMES = ["Сировина", "Машинний час", "Робочий час"]

print(f"Z = {c[0]:g}*x1 + {c[1]:g}*x2 -> max")
for i in range(m):
    print(f"  {A[i,0]:g}*x1 + {A[i,1]:g}*x2 <= {b[i]:g}   ({RESOURCE_NAMES[i]})")
print("  x1, x2 >= 0")


## 2. Канонічна форма

Кожну нерівність доповнюємо невід'ємною залишковою змінною s_i — невикористаним залишком i-го ресурсу: a_i1*x1 + a_i2*x2 + s_i = b_i.

In [ ]:
for i in range(m):
    print(f"  {A[i,0]:g}*x1 + {A[i,1]:g}*x2 + s{i+1} = {b[i]:g},  s{i+1} >= 0")


## 3. Графічний розв'язок

Допустима область — опуклий многокутник (перетин трьох напівплощин і першого квадранта). Вершини шукаємо перебором попарних перетинів меж (трьох обмежень і двох осей) з відсівом недопустимих точок; оптимум — вершина, де лінія рівня Z=const востаннє торкається області в напрямку градієнта c.

In [ ]:
def polygon_vertices(A, b):
    lines = [(A[i, 0], A[i, 1], b[i]) for i in range(len(b))]
    lines.append((1.0, 0.0, 0.0))  # x1 = 0
    lines.append((0.0, 1.0, 0.0))  # x2 = 0
    pts = []
    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            a1, a2, r1 = lines[i]
            a3, a4, r2 = lines[j]
            M = np.array([[a1, a2], [a3, a4]])
            if abs(np.linalg.det(M)) < 1e-9:
                continue
            p = np.linalg.solve(M, [r1, r2])
            if p[0] < -1e-6 or p[1] < -1e-6:
                continue
            if np.all(A @ p <= b + 1e-6):
                pts.append((round(float(p[0]), 6), round(float(p[1]), 6)))
    pts = sorted(set(pts))
    pts = np.array(pts)
    centroid = pts.mean(axis=0)
    angles = np.arctan2(pts[:, 1] - centroid[1], pts[:, 0] - centroid[0])
    return pts[np.argsort(angles)]

poly = polygon_vertices(A, b)
print("Вершини допустимої області:", poly)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.fill(poly[:, 0], poly[:, 1], color="tab:blue", alpha=0.25, label="Допустима область")
ax.plot(np.append(poly[:, 0], poly[0, 0]), np.append(poly[:, 1], poly[0, 1]), "b-")
lim_x = poly[:, 0].max() * 1.3 + 1
lim_y = poly[:, 1].max() * 1.3 + 1
xs_line = np.linspace(0, lim_x, 200)
for i in range(m):
    a1, a2 = A[i]
    ys_line = (b[i] - a1 * xs_line) / a2
    ax.plot(xs_line, ys_line, "--", lw=1, label=f"{RESOURCE_NAMES[i]}")
for px, py in poly:
    ax.plot(px, py, "ko", ms=5)
ax.set_xlim(0, lim_x); ax.set_ylim(0, lim_y)
ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.legend(fontsize=8)
ax.set_title(f"Варіант {VARIANT}: допустима область")
fig.tight_layout()


## 4. Ручний розв'язок симплекс-методом (усі таблиці)

Початковий базис — залишкові змінні s1, s2, s3 (вершина x1=x2=0). На кожній ітерації: (1) розв'язувальний стовпець — змінна з найбільш від'ємною оцінкою Δj у Z-рядку; (2) розв'язувальний рядок — мінімальне симплексне відношення b_i/a_ik серед додатних a_ik; (3) перерахунок методом Жордана-Гауса.

In [ ]:
def simplex_solve(c, A, b, max_iter=30):
    """Табличний симплекс-метод для max c^T x, Ax <= b, x >= 0, b >= 0."""
    mm, nn = A.shape
    T = np.hstack([A, np.eye(mm), b.reshape(-1, 1)]).astype(float)
    obj = np.concatenate([-c, np.zeros(mm + 1)])      # Z-рядок: Z - c^T x = 0
    basis = list(range(nn, nn + mm))                   # початковий базис -- слеки
    var_names = [f"x{i+1}" for i in range(nn)] + [f"s{i+1}" for i in range(mm)]

    tableaus = []
    while True:
        snap = {"basis": list(basis), "T": T.copy(), "obj": obj.copy(), "pivot": None}
        if np.all(obj[:-1] >= -1e-9):                  # критерій оптимальності
            tableaus.append(snap)
            break
        col = int(np.argmin(obj[:-1]))                 # розв'язувальний стовпець (правило Данціга)
        ratios = [T[i, -1] / T[i, col] if T[i, col] > 1e-9 else np.inf for i in range(mm)]
        row = int(np.argmin(ratios))                   # розв'язувальний рядок (мін. відношення)
        snap["pivot"] = {"entering": var_names[col], "leaving": var_names[basis[row]]}
        tableaus.append(snap)

        pivot = T[row, col]
        T[row, :] /= pivot                              # нормування розв'язувального рядка
        for i in range(mm):
            if i != row:
                T[i, :] -= T[i, col] * T[row, :]        # виключення Жордана-Гауса
        obj -= obj[col] * T[row, :]
        basis[row] = col
    x = np.zeros(nn)
    for i, bidx in enumerate(basis):
        if bidx < nn:
            x[bidx] = T[i, -1]
    return x, float(obj[-1]), basis, var_names, tableaus


In [ ]:
x_manual, Z_manual, basis_final, var_names, tableaus = simplex_solve(c, A, b)

for k, snap in enumerate(tableaus):
    basis_names = [var_names[i] for i in snap["basis"]]
    df = pd.DataFrame(
        np.vstack([snap["T"], snap["obj"]]),
        columns=var_names + ["B"],
        index=basis_names + ["Delta_j"],
    )
    print(f"--- Таблиця {k} ---")
    print(df.round(3))
    if snap["pivot"]:
        print(f"  вводимо {snap['pivot']['entering']}, виводимо {snap['pivot']['leaving']}")
    print()

print(f"x* = {x_manual}, Z* = {Z_manual:.4f}")


## 5. Перевірка засобами scipy.optimize.linprog

linprog мінімізує, тому цільову функцію беремо зі знаком «-»: minimize(-c) <=> maximize(c).

In [ ]:
res = linprog(-c, A_ub=A, b_ub=b, bounds=[(0, None)] * n, method="highs")
x_scipy = res.x
Z_scipy = float(-res.fun)
slacks_scipy = res.slack
duals_scipy = -res.ineqlin.marginals

print("SciPy/HiGHS: x* =", x_scipy, " Z* =", Z_scipy)
print("залишки ресурсів:", slacks_scipy)
print("двоїсті оцінки :", duals_scipy)
print("Збіг із ручним симплексом:", np.allclose(x_manual, x_scipy, atol=1e-3) and abs(Z_manual - Z_scipy) < 1e-2)


## 6. Двоїста задача та перевірка теорем двоїстості

W = b1*y1 + b2*y2 + b3*y3 -> min за A^T*y >= c, y >= 0. Двоїсті оцінки y* зчитуємо прямо з фінальної симплекс-таблиці прямої задачі — це коефіцієнти Z-рядка у стовпцях залишкових змінних s1, s2, s3.

In [ ]:
final_obj = tableaus[-1]["obj"]
y_manual = np.array([final_obj[n + i] for i in range(m)])
W_manual = float(b @ y_manual)

print("y* =", y_manual, " W* =", W_manual)
print("Перша теорема двоїстості (Z* = W*):", abs(Z_manual - W_manual) < 1e-3)


## 7. Доповнювальна нежорсткість, дефіцитні та надлишкові ресурси

In [ ]:
rows = []
for i in range(m):
    used = float(A[i] @ x_manual)
    slack_i = float(b[i] - used)
    status = "дефіцитний" if slack_i < 1e-6 else "надлишковий"
    rows.append(dict(ресурс=RESOURCE_NAMES[i], запас=float(b[i]), використано=used,
                      залишок=slack_i, тіньова_ціна=float(y_manual[i]), статус=status))
pd.DataFrame(rows)


## 8. Аналіз чутливості: залежність Z* від запасу кожного ресурсу (±25 %)

In [ ]:
pct_grid = [-25, -12.5, 0, 12.5, 25]
sens = {i: [] for i in range(m)}
for i in range(m):
    for pct in pct_grid:
        b_mod = b.copy()
        b_mod[i] = b[i] * (1 + pct / 100.0)
        r = linprog(-c, A_ub=A, b_ub=b_mod, bounds=[(0, None)] * n, method="highs")
        sens[i].append((pct, float(b_mod[i]), float(-r.fun)))

fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4.5))
for i, ax2 in enumerate(axes2):
    bs = [row[1] for row in sens[i]]
    zs = [row[2] for row in sens[i]]
    ax2.plot(bs, zs, "o-")
    ax2.axvline(float(b[i]), color="gray", ls=":")
    ax2.set_title(RESOURCE_NAMES[i]); ax2.set_xlabel(f"b{i+1}"); ax2.set_ylabel("Z*")
fig2.suptitle(f"Варіант {VARIANT}: Z*(b_i), ±25%")
fig2.tight_layout()


## 9-10. Інтервали стійкості двоїстих оцінок та управлінський висновок

Межі зміни b_i, що не змінюють оптимальний базис, знаходимо з фінальної симплекс-таблиці: стовпець B^-1 для ресурсу i — це стовпець слекової змінної s_i у фінальній таблиці; вимагаємо, щоб усі базисні значення x_B + delta*стовпець лишались невід'ємними.

In [ ]:
def rhs_stability_interval(T, slack_col_idx, b_i, n):
    col = T[:, n + slack_col_idx]
    xB = T[:, -1]
    lower_cands, upper_cands = [], []
    for i in range(len(col)):
        if col[i] > 1e-9:
            lower_cands.append(-xB[i] / col[i])
        elif col[i] < -1e-9:
            upper_cands.append(-xB[i] / col[i])
    delta_min = max(lower_cands) if lower_cands else -np.inf
    delta_max = min(upper_cands) if upper_cands else np.inf
    return b_i + delta_min, b_i + delta_max

final_T = tableaus[-1]["T"]
stab_rows = []
for i in range(m):
    lo, hi = rhs_stability_interval(final_T, i, float(b[i]), n)
    stab_rows.append(dict(ресурс=RESOURCE_NAMES[i], b_min=lo, b_поточне=float(b[i]), b_max=hi,
                           тіньова_ціна=float(y_manual[i])))
stab_df = pd.DataFrame(stab_rows)
print(stab_df)

best_i = int(np.argmax(y_manual))
print(f"\nПершим доцільно закуповувати ресурс «{RESOURCE_NAMES[best_i]}»"
      f" за ціною не вище {float(y_manual[best_i]):.4f} грн/од (його тіньова ціна).")
